# 04 - QLoRA模型微调训练

在DGX Spark上使用QLoRA技术对基座模型进行TRIZ领域微调。

**硬件要求**: DGX Spark (128GB Unified Memory) | 训练时间: ~15小时/epoch

## 4.1 加载配置和数据

In [ ]:
# 创建Trainer (使用formatting_func，不传data_collator)
from utils.training_utils import create_trainer
from config import DATA_CONFIG

trainer = create_trainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    training_args=training_args,
    system_message=DATA_CONFIG['chatml']['system_message'],
    max_seq_length=DATA_CONFIG['chatml']['max_length'],
    packing=True,
)

print('Trainer创建完成，开始训练...')
print('='*60)

# 开始训练
trainer.train()

print('='*60)
print('训练完成!')

## 4.2 加载模型 (4-bit量化)

In [ ]:
# 模型路径
model_path = os.path.join(MODELS_DIR, BASE_MODEL.split('/')[-1])

print(f"加载模型: {model_path}")
print("启用4-bit量化以节省内存...")

# 加载4-bit量化模型
model, tokenizer = load_model_and_tokenizer(
    model_name_or_path=model_path,
    quantization_config=QLORA_CONFIG['quantization'],
    device_map='auto',
    trust_remote_code=True,
)

print("\n模型加载完成!")
print(f"显存占用: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

## 4.3 配置QLoRA

## 4.3 检测模型模块 (可选验证)

Qwen3.6使用混合架构(Gated DeltaNet + Gated Attention + MoE)，模块名称与传统Transformer不同。运行以下代码自动检测实际模块名，确保target_modules配置正确。

In [ ]:
# (可选) 自动检测模型中的线性层模块名
# 用于验证 target_modules 配置是否正确
from utils.training_utils import find_all_linear_names

print("正在检测模型中的线性层模块...")
detected_modules = find_all_linear_names(model)
print(f"\n检测到 {len(detected_modules)} 个可用于LoRA的模块:")
for m in detected_modules:
    print(f"  - {m}")

# 与推荐列表对比
from utils.training_utils import get_qwen36_target_modules
recommended = get_qwen36_target_modules()
missing = set(recommended) - set(detected_modules)
extra = set(detected_modules) - set(recommended)

if missing:
    print(f"\n警告: 推荐列表中有但模型中未检测到: {missing}")
if extra:
    print(f"\n提示: 模型中存在额外模块: {extra}")
if not missing and not extra:
    print("\n模块列表匹配完美!")

In [ ]:
# 创建LoRA配置
lora_config = setup_qlora_config(
    r=QLORA_CONFIG['lora']['r'],
    lora_alpha=QLORA_CONFIG['lora']['lora_alpha'],
    target_modules=QLORA_CONFIG['lora']['target_modules'],
    lora_dropout=QLORA_CONFIG['lora']['lora_dropout'],
    use_rslora=QLORA_CONFIG['lora'].get('use_rslora', False),
)

# 准备QLoRA模型
model = prepare_qlora_model(model, lora_config)

print("\nQLoRA配置完成!")

## 4.4 配置训练参数

In [ ]:
# 训练参数
training_args = setup_training_arguments(
    output_dir=QLORA_CONFIG['training']['output_dir'],
    num_train_epochs=QLORA_CONFIG['training']['num_train_epochs'],
    per_device_batch_size=QLORA_CONFIG['training']['per_device_train_batch_size'],
    gradient_accumulation_steps=QLORA_CONFIG['training']['gradient_accumulation_steps'],
    learning_rate=QLORA_CONFIG['training']['learning_rate'],
    warmup_ratio=QLORA_CONFIG['training']['warmup_ratio'],
    save_steps=QLORA_CONFIG['training']['save_steps'],
    eval_steps=QLORA_CONFIG['training']['eval_steps'],
    logging_steps=QLORA_CONFIG['training']['logging_steps'],
)

print("训练参数配置完成!")

## 4.5 创建Trainer并开始训练

In [ ]:
# 创建Trainer
trainer = create_trainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    training_args=training_args,
    max_seq_length=DATA_CONFIG['chatml']['max_length'],
    packing=True,
)

print("Trainer创建完成，开始训练...")
print("="*60)

# 开始训练
trainer.train()

print("="*60)
print("训练完成!")

## 4.6 保存模型

In [ ]:
from utils.training_utils import save_adapter_only

# 保存LoRA适配器 (仅100-200MB)
adapter_output_dir = os.path.join(MODELS_DIR, 'meerkat_triz_adapter_v1')
save_adapter_only(model, tokenizer, adapter_output_dir)

print(f"\n适配器已保存到: {adapter_output_dir}")

# 查看文件大小
import os
for f in os.listdir(adapter_output_dir):
    fp = os.path.join(adapter_output_dir, f)
    size = os.path.getsize(fp) / 1024**2
    print(f"  {f}: {size:.2f} MB")

## 4.7 合并并保存完整模型 (可选)

合并LoRA适配器与基座模型，生成完整的FP16模型。
**注意**: 合并后的模型约140GB，确保磁盘空间充足。

In [ ]:
# 合并模型 (需要大量内存，可选步骤)
# from utils.training_utils import merge_and_save_model
#
# merged_output = os.path.join(MODELS_DIR, 'meerkat_triz_merged_v1')
# merge_and_save_model(
#     base_model_path=model_path,
#     adapter_path=adapter_output_dir,
#     output_path=merged_output,
# )

print("跳过合并步骤 (可在需要时单独运行)")
print("LoRA适配器可直接用于推理，无需合并")

## 4.8 清理显存

In [ ]:
# 清理显存
del model
del tokenizer
del trainer
torch.cuda.empty_cache()

print("显存已清理")
print(f"当前显存占用: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

---

## 下一步

微调完成！请打开: **05_model_evaluation.ipynb** 评估微调效果